In [1]:
import pandas as pd
import numpy as np
import glob
import pickle

### get the conversion from volts to degree from the previously computed calibration values 

In [2]:
motor_cal = pd.read_csv('./dataFolder/2b_OtherFilesNeededForAnalysis/motorcalibrationvalues.csv')
m = motor_cal.loc[motor_cal['parameter'] == 'slope_m', 'value'].values[0]
b = motor_cal.loc[motor_cal['parameter'] == 'intercept_b', 'value'].values[0]
print(f'Motor calibration: m={m:.4f} cm/V, b={b:.4f} cm')

Motor calibration: m=-0.1176 cm/V, b=-0.0027 cm


In [3]:
out_path = "./dataFolder/4_RawData_Index_Timestamp/ScaledStimulus/"
all_files_rawV = glob.glob('./dataFolder/4_RawData_Index_Timestamp/*.pickle')
all_files_scaledV = glob.glob(out_path + '*.pickle')
print(f'Found {len(all_files_scaledV)} files')

Found 4 files


In [4]:
# save the data on distance to the motor from pilifer as a csv for future use

attachment_distances = pd.DataFrame({
    'moth': ['m23_wn2', 'm29_wn2', 'm30_wn1', 'm31_wn2'],
    'attachment_distance_cm': [0.7, 0.7, 0.8, 0.7]})
attachment_distances.to_csv(out_path + '/moth_attachment_distances.csv', index=False)

print(attachment_distances)

# convert this to a dict
attachment_distances = attachment_distances.set_index('moth')['attachment_distance_cm'].to_dict()

      moth  attachment_distance_cm
0  m23_wn2                     0.7
1  m29_wn2                     0.7
2  m30_wn1                     0.8
3  m31_wn2                     0.7


In [5]:
# get all files for which we have data

for moth in attachment_distances.keys():
        
    filepath = [file for file in all_files_scaledV if moth in file][0]
    with open(filepath, "rb") as f:
        data = pickle.load(f)    
    scaled_V = data[2]
    displacement_cm = m * scaled_V + b
    
    attachment_distance_cm = attachment_distances[moth]  
    angle_degrees = np.degrees(np.arctan(displacement_cm / attachment_distance_cm))
    
    #get raw V to be put in df for checking
    rawfilepath = [file for file in all_files_rawV if moth in file][0]
    with open(rawfilepath, "rb") as f:
        rawdata = pickle.load(f)    
    raw_V = rawdata[2]
    
    result_df = pd.DataFrame({
        'raw_voltage': raw_V,
        'scaled_voltage': scaled_V,
        'displacement_cm': displacement_cm,
        'deflectionAngle_degrees': angle_degrees
    })
    result_df.to_csv(out_path + moth + '_DisplacementIndegrees.csv', index=False)
    print(f'{moth}: angle range {angle_degrees.min():.2f}° to {angle_degrees.max():.2f}°')
    
    #also create a new pickle dump for disp in degree along with timestamp and index
    # Save individually per moth
    all_Units_aligned = data[0]
    all_Units_prior = data[1]
    WhiteNoise_deg = angle_degrees
    saveObject = (all_Units_aligned,all_Units_prior, WhiteNoise_deg)
    with open(out_path + moth + '_timeStamps_degrees.pickle',"wb") as f:
        pickle.dump(saveObject, f)

m23_wn2: angle range -26.38° to 31.32°
m29_wn2: angle range -21.42° to 23.80°
m30_wn1: angle range -23.80° to 28.19°
m31_wn2: angle range -21.34° to 23.95°
